# قِرْشَك | Qirshak — Smart Spending Watcher

مساعد مالي متعدد الوكلاء (CrewAI) يراقب صرفك، يكتشف الأنماط، وينبهك قبل ما تخسر السيطرة على ميزانيتك.

**بنية النوتبوك:**
1. الإعداد المشترك (الكل يشغّله أول)
2. المجموعة 1 — Categorizer + Pattern Detective
3. المجموعة 2 — Time-Based Alert + Predictor
4. المجموعة 3 — Budget & Savings Advisor + Tone Agent
5. تجميع الفريق الكامل (Crew) — يُكمّل بعد جهوزية كل المجموعات
6. التشغيل والاختبار النهائي

⚠️ كل عضو ينسخ القسم 1 كما هو بدون تعديل، ويشتغل على قسمه المخصص فقط.

# Section 1: Shared Setup

Everyone runs these four cells exactly as they are. Do not edit anything except your personal API key.

In [9]:
# 1. Install
!pip install -q -U crewai pandas

In [2]:
# 2. Imports
import os
import pandas as pd
from crewai import Agent, Task, Crew, Process, LLM

In [ ]:
# 3. Configure the LLM (each member enters their own key here at run time)
os.environ["OPENROUTER_API_KEY"] = ""

llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

In [5]:
# 4. Load the bank statement data (same file for everyone)

import urllib.request

url = "https://raw.githubusercontent.com/ramahIO/Qirshak/refs/heads/main/data/mock_statement.csv"
urllib.request.urlretrieve(url, "mock_statement.csv")

statement_df = pd.read_csv("mock_statement.csv")
statement_text = statement_df.to_string(index=False)

print("Statement loaded:", len(statement_df), "transactions")

Statement loaded: 43 transactions


In [14]:
# التحقق الآلي من المجاميع (Ground Truth) — يُستخدم لاحقاً كمرجع تحقق
category_map = {
    "STARBUCKS COFFEE JEDDAH": "coffee", "BARNS COFFEE": "coffee", "DR CAFE COFFEE": "coffee",
    "JAHEZ DELIVERY": "delivery", "HUNGERSTATION": "delivery",
    "NETFLIX.COM": "subscriptions", "SHAHID VIP": "subscriptions", "SPOTIFY PREMIUM": "subscriptions",
    "UBER TRIP": "transport", "CAREEM RIDE": "transport",
    "PANDA HYPERMARKET": "groceries", "CARREFOUR MARKET": "groceries",
    "RENT PAYMENT TRANSFER": "bills_rent", "STC MOBILE BILL": "bills_rent",
    "ELECTRICITY BILL SEC": "bills_rent", "STC PAY TRANSFER": "bills_rent",
    "SALARY - COMPANY XYZ": "income"
}

statement_df["category_verified"] = statement_df["description"].map(category_map)
true_totals = statement_df.groupby("category_verified")["amount_sar"].sum().round(2)

print("VERIFIED TOTALS (calculated directly from data, not by the LLM):")
print(true_totals)

VERIFIED TOTALS (calculated directly from data, not by the LLM):
category_verified
bills_rent      -1900.00
coffee           -435.50
delivery         -348.00
groceries        -973.50
income           4000.00
subscriptions     -88.98
transport         -90.50
Name: amount_sar, dtype: float64


In [24]:
verified_summary = f"""
VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):

Income: {true_totals['income']:.2f} SAR (received on day 1 of the month)
Coffee: {true_totals['coffee']:.2f} SAR
Delivery: {true_totals['delivery']:.2f} SAR
Groceries: {true_totals['groceries']:.2f} SAR
Bills & Rent: {true_totals['bills_rent']:.2f} SAR
Subscriptions: {true_totals['subscriptions']:.2f} SAR
Transport: {true_totals['transport']:.2f} SAR

Total expenses: {true_totals.drop('income').sum():.2f} SAR
Remaining balance: {true_totals.sum():.2f} SAR

Statement covers a 30-day month. Last recorded transaction was on day 27.
No transactions were recorded for days 28-30 (balance nearly depleted).
"""

print(verified_summary)


VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):

Income: 4000.00 SAR (received on day 1 of the month)
Coffee: -435.50 SAR
Delivery: -348.00 SAR
Groceries: -973.50 SAR
Bills & Rent: -1900.00 SAR
Subscriptions: -88.98 SAR
Transport: -90.50 SAR

Total expenses: -3836.48 SAR
Remaining balance: 163.52 SAR

Statement covers a 30-day month. Last recorded transaction was on day 27.
No transactions were recorded for days 28-30 (balance nearly depleted).



In [15]:
print(statement_df.head(10))

         date   time              description  amount_sar category_verified
0  2026-07-01  09:00     SALARY - COMPANY XYZ     4000.00            income
1  2026-07-01  17:15  STARBUCKS COFFEE JEDDAH      -22.00            coffee
2  2026-07-02  08:30        PANDA HYPERMARKET     -186.50         groceries
3  2026-07-02  18:00           JAHEZ DELIVERY      -54.00          delivery
4  2026-07-03  11:00          STC MOBILE BILL     -120.00        bills_rent
5  2026-07-03  16:45             BARNS COFFEE      -19.00            coffee
6  2026-07-03  20:10                UBER TRIP      -28.50         transport
7  2026-07-04  17:30           DR CAFE COFFEE      -24.00            coffee
8  2026-07-05  09:00    RENT PAYMENT TRANSFER    -1500.00        bills_rent
9  2026-07-05  10:00              NETFLIX.COM      -39.99     subscriptions


---
# Section 2: Group 1 — Categorizer + Pattern Detective

**Receives:** `statement_text` directly from Section 1

**Delivers:** a text report with (a) every transaction categorized, (b) a list of detected patterns

In [16]:
categorizer = Agent(
    role="Personal Spending Categorizer",

    goal=(
        "Classify every transaction in a bank statement into one clear "
        "category, using only the transaction description and amount "
        "provided — never guessing beyond what the data shows."
    ),

    backstory=(
        "You are a meticulous financial analyst who reads raw bank "
        "statement rows and assigns each one to exactly one category: "
        "coffee, delivery, subscriptions, transport, groceries, "
        "bills_rent, income, or other. You never invent a transaction "
        "that isn't in the data, and you never assign two categories "
        "to the same row."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [17]:
# --- Agent 2: Pattern Detective ---
pattern_detective = Agent(
    role="Spending Pattern Detective",

    goal=(
        "Analyze categorized spending data to surface behavioral patterns "
        "the person may not have noticed themselves — repetition, "
        "overlapping subscriptions, and timing trends — based only on "
        "the data provided."
    ),

    backstory=(
        "You are a behavioral finance analyst who specializes in finding "
        "quiet spending leaks: small repeated purchases that add up, "
        "subscriptions that overlap in purpose, and differences between "
        "weekday and weekend spending. You report only patterns that are "
        "clearly supported by the data — you never claim a subscription "
        "is 'unused' unless the data actually shows that, since usage "
        "isn't something a bank statement can confirm."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [18]:
# --- Task 1: Categorization Task ---
categorization_task = Task(
    description=f'''
Categorize every transaction in the bank statement below into exactly
one of these categories: coffee, delivery, subscriptions, transport,
groceries, bills_rent, income, other.

{statement_text}

Rules:

- Use only the transaction description and amount provided.
- Do not invent transactions that are not in the data.
- Assign exactly one category per transaction — never two.
- Treat any positive amount as income.
- Group your output by category, and for each category state:
  the transactions in it, the count, and the total amount.
- Do not skip any transaction from the statement.
''',
    expected_output=(
        "A structured report grouping every transaction into its "
        "category, with a running total (SAR) and transaction count "
        "for each category."
    ),
    agent=categorizer
)

In [19]:
# --- Task 2: Pattern Detection Task ---
pattern_task = Task(
    description='''
Based on the categorized spending report above, identify behavioral
spending patterns.

Look specifically for:

1. Repeated small purchases in the same category (e.g. frequent coffee
   purchases) — note the frequency and typical time of day if visible
   in the data.
2. Subscriptions or recurring charges that fall into the same category
   (e.g. multiple streaming services) — flag them as worth reviewing,
   without claiming any of them are unused.
3. Any noticeable difference between spending on different days of the
   week, if the data shows one.

Rules:

- Base every pattern strictly on the categorized data — do not invent
  a pattern the data does not support.
- Do not make claims about whether a subscription is used or unused.
- If a pattern is weak or uncertain, say so explicitly rather than
  presenting it as a strong finding.
''',
    expected_output=(
        "A short list of clearly supported spending patterns, each with "
        "the evidence behind it (frequency, amount, or timing), and any "
        "patterns explicitly flagged as uncertain."
    ),
    agent=pattern_detective,
    context=[categorization_task]
)

In [20]:
# --- Test Group 1 on its own (mini Crew, no need to wait for other groups) ---
group1_crew = Crew(
    agents=[categorizer, pattern_detective],
    tasks=[categorization_task, pattern_task],
    process=Process.sequential,
    verbose=True
)

group1_result = await group1_crew.kickoff_async()
print(group1_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5b72873f-e082-43f6-83b1-7f57e80aa5f7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00                                                           │
│  2026-07-13 18:00           HUNGERSTATION      -58.00  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Spending Categorizer                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00  

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Spending Categorizer                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Coffee Transactions:**                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -22.00                                                                              │
│  - BARNS COFFEE: -19.00                                                                                         │
│  - DR CAFE COFFEE: -24.00                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -21.00                                                                              │
│  - STARBUCKS COFFEE JEDDAH: -25.00                                                                              │
│  - BARNS COFFEE: -18.50                                                                                         │
│  - STARBUCKS COFFEE JEDDAH: -23.00                                                                              │
│  - DR CAFE COFFEE: -20.00                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -24.00                                                                              │
│  - DR CAFE COFFEE: -22.50                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -24.00                                                                              │
│  - BARNS COFFEE: -19.50                                                                                         │
│  - STARBUCKS COFFEE JEDDAH: -26.00                                                                              │
│  - DR CAFE COFFEE: -21.00                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -22.50                                                                              │
│  - BARNS COFFEE: -19.00                                                                                         │
│  - STARBUCKS COFFEE JEDDAH: -23.50                                                                              │
│  - DR CAFE COFFEE: -20.50                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -24.50                                                                              │
│  - STARBUCKS COFFEE JEDDAH: -22.00                                                                              │
│                                                                                                                 │
│  **Count:** 20                                                                                                  │
│  **Total Amount:** -426.00 SAR                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Delivery Transactions:**                                                                                     │
│  - JAHEZ DELIVERY: -54.00                                                                                       │
│  - HUNGERSTATION: -63.00                               

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Pattern Detective                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. **Repeated Small Purchases in Coffee Category:**                                                            │
│     - There are 20 transactions related to coffee, totaling -426.00 SAR.                                        │
│     - The frequency of purchases indicates a strong pattern of regular coffee consumption.                      │
│     - Notable purchases include multiple transactions at STARBUCKS COFFEE JEDDAH, with 10 occurrences, and DR   │
│  CAFE COFFEE, with 5 occurrences.                                                                               │
│     - The amounts spent range from -18.50 SAR to -26.00 SAR, suggesting a consistent spending behavior on       │
│  coffee.                                                                                                        │
│                                                                                                                 │
│  2. **Subscriptions or Recurring Charges:**                                                                     │
│     - There are 3 subscription transactions totaling -88.98 SAR:                                                │
│       - NETFLIX.COM: -39.99 SAR                                                                                 │
│       - SHAHID VIP: -29.00 SAR                                                                                  │
│       - SPOTIFY PREMIUM: -19.99 SAR                                                                             │
│     - These subscriptions fall into the entertainment category, and while they serve different purposes, they   │
│  may overlap in content consumption. This could be worth reviewing for potential redundancy.                    │
│                                                                                                                 │
│  3. **Transport Transactions:**                                                                                 │
│     - There are 3 transport-related transactions totaling -90.50 SAR:                                           │
│       - UBER TRIP: -28.50 SAR                                                                                   │
│       - CAREEM RIDE: -32.00 SAR                                                                                 │
│       - CAREEM RIDE: -30.00 SAR                                                                                 │
│     - This indicates a pattern of using ride-sharing services, but the frequency is low, making it a weaker     │
│  finding.                                                                                                       │
│                                                                                                                 │
│  4. **Groceries Spending:**                                                                                     │
│     - There are 6 grocery transactions totaling -1,073.50 SAR, with significant amounts spent at PANDA          │
│  HYPERMARKET and CARREFOUR MARKET.                                                                              │
│     - The amounts range from -95.00 SAR to -210.00 SAR, indicating a consistent pattern of grocery shopping,    │
│  but without specific timing data, the frequency of these purchases is less clear.                              │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Based on the categorized spending report above, identify behavioral                                            │
│  spending patterns.                                                                                             │
│                                                                                                                 │
│  Look specifically for:                                                                                         │
│                                                                                                                 │
│  1. Repeated small purchases in the same category (e.g. frequent coffee                                         │
│     purchases) — note the frequency and typical time of day if visible                                          │
│     in the data.                                                                                                │
│  2. Subscriptions or recurring charges that fall into the same category                                         │
│     (e.g. multiple streaming services) — flag them as worth reviewing,                                          │
│     without claiming any of them are unused.                                                                    │
│  3. Any noticeable difference between spending on different days of the                                         │
│     week, if the data shows one.                                                                                │
│                                                                                                                 │
│  Rules:                                                                                                         │
│                                                                                                                 │
│  - Base every pattern strictly on the categorized data — do not invent                                          │
│    a pattern the data does not support.                                                                         │
│  - Do not make claims about whether a subscription is used or unused.                                           │
│  - If a pattern is weak or uncertain, say so explicitly rather than                                             │
│    presenting it as a strong finding.                                                                           │
│                                                                                                                 │
│  Agent: Spending Pattern Detective                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

1. **Repeated Small Purchases in Coffee Category:**
   - There are 20 transactions related to coffee, totaling -426.00 SAR. 
   - The frequency of purchases indicates a strong pattern of regular coffee consumption. 
   - Notable purchases include multiple transactions at STARBUCKS COFFEE JEDDAH, with 10 occurrences, and DR CAFE COFFEE, with 5 occurrences. 
   - The amounts spent range from -18.50 SAR to -26.00 SAR, suggesting a consistent spending behavior on coffee.

2. **Subscriptions or Recurring Charges:**
   - There are 3 subscription transactions totaling -88.98 SAR:
     - NETFLIX.COM: -39.99 SAR
     - SHAHID VIP: -29.00 SAR
     - SPOTIFY PREMIUM: -19.99 SAR
   - These subscriptions fall into the entertainment category, and while they serve different purposes, they may overlap in content consumption. This could be worth reviewing for potential redundancy.

3. **Transport Transactions:**
   - There are 3 transport-related transactions totaling -90.50 SAR:
     - UBER TRIP: -28.

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5b72873f-e082-43f6-83b1-7f57e80aa5f7                                                                       │
│  Final Output: 1. **Repeated Small Purchases in Coffee Category:**                                              │
│     - There are 20 transactions related to coffee, totaling -426.00 SAR.                                        │
│     - The frequency of purchases indicates a strong pattern of regular coffee consumption.                      │
│     - Notable purchases include multiple transactions at STARBUCKS COFFEE JEDDAH, with 10 occurrences, and DR   │
│  CAFE COFFEE, with 5 occurrences.                                                                               │
│     - The amounts spent range from -18.50 SAR to -26.00 SAR, suggesting a consistent spending behavior on       │
│  coffee.                                                                                                        │
│                                                                                                                 │
│  2. **Subscriptions or Recurring Charges:**                                                                     │
│     - There are 3 subscription transactions totaling -88.98 SAR:                                                │
│       - NETFLIX.COM: -39.99 SAR                                                                                 │
│       - SHAHID VIP: -29.00 SAR                                                                                  │
│       - SPOTIFY PREMIUM: -19.99 SAR                                                                             │
│     - These subscriptions fall into the entertainment category, and while they serve different purposes, they   │
│  may overlap in content consumption. This could be worth reviewing for potential redundancy.                    │
│                                                                                                                 │
│  3. **Transport Transactions:**                                                                                 │
│     - There are 3 transport-related transactions totaling -90.50 SAR:                                           │
│       - UBER TRIP: -28.50 SAR                                                                                   │
│       - CAREEM RIDE: -32.00 SAR                                                                                 │
│       - CAREEM RIDE: -30.00 SAR                                                                                 │
│     - This indicates a pattern of using ride-sharing services, but the frequency is low, making it a weaker     │
│  finding.                                                                                                       │
│                                                                                                                 │
│  4. **Groceries Spending:**                                                                                     │
│     - There are 6 grocery transactions totaling -1,073.50 SAR, with significant amounts spent at PANDA          │
│  HYPERMARKET and CARREFOUR MARKET.                                                                              │
│     - The amounts range from -95.00 SAR to -210.00 SAR, indicating a consistent pattern of grocery shopping,    │
│  but without specific timing data, the frequency of these purchases is less clear.                              │
│                                                       

---
# Section 3: Group 2 — Time-Based Alert + Predictor

**Receives:** Group 1's output (or a temporary placeholder for early independent testing)

**Delivers:** (a) a live alert if triggered, (b) a projection of when the balance will run out before month end

In [25]:
# Temporary placeholder representing the shape of Group 1's output
# (replaced later with context=[categorization_task, pattern_task] at merge time)
placeholder_categorized_data = """
Categories found: Coffee (15 transactions, avg 22 SAR, mostly between 16:00-19:00),
Subscriptions (Netflix, Shahid, Spotify - all charged same day, same category),
Groceries (4 transactions), Rent (1500 SAR), Bills (300 SAR total),
Transport (3 transactions), Income: 4000 SAR salary on day 1.
"""

In [ ]:
# --- Agent 3: Time-Based Alert Agent ---
time_alert_agent = Agent(
    role="TODO",
    goal="TODO",
    backstory="TODO",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [26]:
# --- Agent 4: Predictor ---
predictor = Agent(
    role="Spending Predictor",

    goal=(
        "Project forward from verified monthly spending totals to estimate "
        "whether the person will run short of money before the end of the "
        "month, and by how many days, using only the numbers provided."
    ),

    backstory=(
        "You are a careful financial forecaster. You work only with "
        "verified totals — you never recalculate or second-guess the "
        "numbers you're given, and you never invent spending figures. "
        "Your job is to reason about the pace of spending relative to "
        "income and the number of days in the month."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Task 3: Time-Based Alert Task ---
alert_task = Task(
    description=f'''
TODO: Based on the time pattern below, decide whether the current (simulated)
time warrants an alert:

{placeholder_categorized_data}

TODO: add a "current simulated time" variable and a confidence threshold rule
''',
    expected_output="TODO",
    agent=time_alert_agent
)

In [29]:
# --- Task 4: Prediction Task ---
prediction_task = Task(
    description=f'''
Using the verified spending summary below, project whether the person's
balance will run out before the end of the month, and if so, by how many
days.

{verified_summary}

You MUST follow this exact structure in your answer, with real numbers
filled in at each step — do not skip any step or merge them together:

Step 1 - Daily spending rate:
State the total expenses and the number of days covered (day 1 to day 27),
then divide to get the average daily spending rate. Show the division.

Step 2 - Days the remaining balance can cover:
Divide the remaining balance by the daily spending rate from Step 1.
Show the division and the resulting number of days.

Step 3 - Final projection:
State the exact day number (out of 30) when the balance is expected to
run out, and how many days before month-end that is.

Rules:

- Use only the numbers provided above — do not recalculate category
  totals or invent new figures.
- Every step must show the actual arithmetic (the numbers and the
  operation), not just the final result.
- If the projection is uncertain (e.g. spending pace may vary), say so
  after Step 3, not instead of it.
''',
    expected_output=(
        "A forecast with three clearly labeled steps (Step 1, Step 2, "
        "Step 3), each showing the arithmetic used, ending with the exact "
        "day the balance is projected to run out and the number of days "
        "before month-end."
    ),
    agent=predictor
)

In [30]:
predictor_test_crew = Crew(
    agents=[predictor],
    tasks=[prediction_task],
    process=Process.sequential,
    verbose=True
)

predictor_result = await predictor_test_crew.kickoff_async()
print(predictor_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e3866b7d-56c1-4509-9bad-42d67fbb1c18                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Using the verified spending summary below, project whether the person's                                        │
│  balance will run out before the end of the month, and if so, by how many                                       │
│  days.                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  You MUST follow this exact structure in your answer, with real numbers                                         │
│  filled in at each step — do not skip any step or merge them together:                                          │
│                                                                                                                 │
│  Step 1 - Daily spending rate:                                                                                  │
│  State the total expenses and the number of days covered (day 1 to day 27),                                     │
│  then divide to get the average daily spending rate. Show the division.                                         │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:                                                                 │
│  Divide the remaining balance by the daily spending rat

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Using the verified spending summary below, project whether the person's                                        │
│  balance will run out before the end of the month, and if so, by how many                                       │
│  days.                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  You MUST follow this exact structure in your answer, with real numbers                                         │
│  filled in at each step — do not skip any step or merge them together:                                          │
│                                                                                                                 │
│  Step 1 - Daily spending rate:                                                                                  │
│  State the total expenses and the number of days covered (day 1 to day 27),                                     │
│  then divide to get the average daily spending rate. Show the division.                                         │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Predictor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Step 1 - Daily spending rate:                                                                                  │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Number of days covered: 27 days                                                                                │
│  Daily spending rate = Total expenses / Number of days covered                                                  │
│  Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)                                     │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:                                                                 │
│  Remaining balance: 163.52 SAR                                                                                  │
│  Daily spending rate: -142.06 SAR                                                                               │
│  Days the remaining balance can cover = Remaining balance / Daily spending rate                                 │
│  Days the remaining balance can cover = 163.52 SAR / 142.06 SAR ≈ 1.15 days                                     │
│                                                                                                                 │
│  Step 3 - Final projection:                                                                                     │
│  The balance is expected to run out on day 28 (1.15 days from day 27).                                          │
│  This means the balance will run out 2 days before month-end (day 30).                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the verified spending summary below, project whether the person's                                        │
│  balance will run out before the end of the month, and if so, by how many                                       │
│  days.                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  You MUST follow this exact structure in your answer, with real numbers                                         │
│  filled in at each step — do not skip any step or merge them together:                                          │
│                                                                                                                 │
│  Step 1 - Daily spending rate:                                                                                  │
│  State the total expenses and the number of days covered (day 1 to day 27),                                     │
│  then divide to get the average daily spending rate. Show the division.                                         │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:                                                                 │
│  Divide the remaining balance by the daily spending rat

Step 1 - Daily spending rate:  
Total expenses: -3836.48 SAR  
Number of days covered: 27 days  
Daily spending rate = Total expenses / Number of days covered  
Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)

Step 2 - Days the remaining balance can cover:  
Remaining balance: 163.52 SAR  
Daily spending rate: -142.06 SAR  
Days the remaining balance can cover = Remaining balance / Daily spending rate  
Days the remaining balance can cover = 163.52 SAR / 142.06 SAR ≈ 1.15 days

Step 3 - Final projection:  
The balance is expected to run out on day 28 (1.15 days from day 27).  
This means the balance will run out 2 days before month-end (day 30).


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e3866b7d-56c1-4509-9bad-42d67fbb1c18                                                                       │
│  Final Output: Step 1 - Daily spending rate:                                                                    │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Number of days covered: 27 days                                                                                │
│  Daily spending rate = Total expenses / Number of days covered                                                  │
│  Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)                                     │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:                                                                 │
│  Remaining balance: 163.52 SAR                                                                                  │
│  Daily spending rate: -142.06 SAR                                                                               │
│  Days the remaining balance can cover = Remaining balance / Daily spending rate                                 │
│  Days the remaining balance can cover = 163.52 SAR / 142.06 SAR ≈ 1.15 days                                     │
│                                                                                                                 │
│  Step 3 - Final projection:                                                                                     │
│  The balance is expected to run out on day 28 (1.15 days from day 27).                                          │
│  This means the balance will run out 2 days before month-end (day 30).                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# --- Test Group 2 on its own ---
group2_crew = Crew(
    agents=[time_alert_agent, predictor],
    tasks=[alert_task, prediction_task],
    process=Process.sequential,
    verbose=True
)

group2_result = await group2_crew.kickoff_async()
print(group2_result.raw)

---
# Section 4: Group 3 — Budget & Savings Advisor + Tone Agent

**Receives:** Groups 1 and 2's output (or a temporary placeholder for early independent testing)

**Delivers:** the final report (budget plan) in Qirshak's friendly tone

In [ ]:
# Temporary placeholder representing the shape of Groups 1 and 2's output
placeholder_full_analysis = """
Categorized spending: Coffee 340 SAR/month, Subscriptions 89 SAR/month,
Groceries 800 SAR/month, Rent 1500 SAR, Bills 300 SAR, Transport 120 SAR.
Prediction: at current spending rate, balance will run out approximately
3-5 days before month end. Income: 4000 SAR.
"""

In [31]:
# --- Agent 5: Budget & Savings Advisor ---
budget_advisor = Agent(
    role="Budget & Savings Advisor",

    goal=(
        "Turn verified spending totals and a shortfall projection into a "
        "concrete daily spending limit per category and a realistic "
        "monthly savings target, grounded in the person's actual income "
        "and behavior rather than generic advice."
    ),

    backstory=(
        "You are a practical financial advisor who never gives generic "
        "advice like 'save 20%.' You build every recommendation from the "
        "specific numbers you are given — actual income, actual category "
        "spending, and the actual shortfall projection. You never invent "
        "a figure that cannot be derived from the data provided."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [32]:
budget_input = f'''
{verified_summary}

SHORTFALL PROJECTION (from the Predictor):
{predictor_result.raw}
'''

print(budget_input)



VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):

Income: 4000.00 SAR (received on day 1 of the month)
Coffee: -435.50 SAR
Delivery: -348.00 SAR
Groceries: -973.50 SAR
Bills & Rent: -1900.00 SAR
Subscriptions: -88.98 SAR
Transport: -90.50 SAR

Total expenses: -3836.48 SAR
Remaining balance: 163.52 SAR

Statement covers a 30-day month. Last recorded transaction was on day 27.
No transactions were recorded for days 28-30 (balance nearly depleted).


SHORTFALL PROJECTION (from the Predictor):
Step 1 - Daily spending rate:  
Total expenses: -3836.48 SAR  
Number of days covered: 27 days  
Daily spending rate = Total expenses / Number of days covered  
Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)

Step 2 - Days the remaining balance can cover:  
Remaining balance: 163.52 SAR  
Daily spending rate: -142.06 SAR  
Days the remaining balance can cover = Remaining balance / Daily spending rate  
Days the remaining balance can cover = 163.52 SAR /

In [36]:
# --- Agent 6: Tone Agent ---
tone_agent = Agent(
    role="Qirshak Tone Rewriter",

    goal=(
        "Rewrite a financial report into Qirshak's warm, friendly, "
        "lightly humorous voice in Arabic — without changing a single "
        "number, fact, or recommendation from the original report."
    ),

    backstory=(
        "You are the voice of Qirshak, a close financial companion who "
        "cares about the user without being preachy or cold. You never "
        "lecture, never guilt-trip, and never sound like a bank statement. "
        "You rewrite tone and wording only — every number, category name, "
        "and recommendation in your output must exactly match the "
        "original report you were given."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [37]:
final_verified_report = f'''
{budget_result.raw}

--- VERIFIED CORRECTION ---
Note: the adjusted total expenses and savings target above may contain a
calculation error. Use these verified figures instead when rewriting:

Verified adjusted total expenses: {adjusted_expenses:.2f} SAR
Verified monthly savings target: {verified_savings_target:.2f} SAR
'''

print(final_verified_report)


**Budget Plan**

**1. Suggested Daily Spending Limit for Discretionary Categories:**

- **Coffee:**
  - Current spending: -435.50 SAR over 27 days
  - Daily average: 435.50 SAR / 27 days ≈ 16.14 SAR
  - Suggested limit: 10 SAR per day
  - Justification: Reducing coffee spending to 10 SAR per day will help cut down on discretionary spending while still allowing for occasional coffee purchases, as current spending is significantly above this limit.

- **Delivery:**
  - Current spending: -348.00 SAR over 27 days
  - Daily average: 348.00 SAR / 27 days ≈ 12.89 SAR
  - Suggested limit: 5 SAR per day
  - Justification: Lowering delivery spending to 5 SAR per day will help manage discretionary expenses more effectively, as current spending is more than double this amount.

- **Transport:**
  - Current spending: -90.50 SAR over 27 days
  - Daily average: 90.50 SAR / 27 days ≈ 3.35 SAR
  - Suggested limit: 3 SAR per day
  - Justification: Setting a limit of 3 SAR per day for transport will hel

In [33]:
# --- Task 5: Budget & Savings Task ---
budget_task = Task(
    description=f'''
Using the verified spending data and shortfall projection below, build a
practical budget plan.

{budget_input}

Your plan must include:

1. A suggested daily spending limit for each discretionary category
   (coffee, delivery, transport) that would reduce the shortfall shown
   above — calculate the reduction needed based on the actual shortfall
   amount, not a generic percentage.
2. A realistic monthly savings target based on the person's actual
   income (4000 SAR) and adjusted spending — not a generic "save 20%"
   rule.
3. A one-sentence explanation for each limit, tied to the specific
   numbers above (e.g. "coffee spending was X SAR over Y transactions").

Rules:

- Every number in your plan must be traceable to the data provided above.
- Do not suggest cutting essential categories (bills_rent, groceries)
  below a reasonable minimum.
- Do not invent figures that cannot be derived from the data given.
''',
    expected_output=(
        "A structured budget plan with a daily limit per discretionary "
        "category, a monthly savings target, and a short justification "
        "for each figure, all traceable to the verified data."
    ),
    agent=budget_advisor
)

In [34]:
budget_test_crew = Crew(
    agents=[budget_advisor],
    tasks=[budget_task],
    process=Process.sequential,
    verbose=True
)

budget_result = await budget_test_crew.kickoff_async()
print(budget_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2e15d64a-5483-4cf3-8b0d-5a4700286d50                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  Step 1 - Daily spending rate:                                                                                  │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Number of days covered: 27 days                                                                                │
│  Daily spending rate = Total expenses / Number of days covered                                                  │
│  Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)                                     │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:                                                                 │
│  Remaining balance: 163.52 SAR                         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget & Savings Advisor                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  Step 1 - Daily spending rate:                                                                                  │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Number of days covered: 27 days                                                                                │
│  Daily spending rate = Total expenses / Number of days covered                                                  │
│  Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)                                     │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:        

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Budget & Savings Advisor                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current spending: -435.50 SAR over 27 days                                                                 │
│    - Daily average: 435.50 SAR / 27 days ≈ 16.14 SAR                                                            │
│    - Suggested limit: 10 SAR per day                                                                            │
│    - Justification: Reducing coffee spending to 10 SAR per day will help cut down on discretionary spending     │
│  while still allowing for occasional coffee purchases, as current spending is significantly above this limit.   │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current spending: -348.00 SAR over 27 days                                                                 │
│    - Daily average: 348.00 SAR / 27 days ≈ 12.89 SAR                                                            │
│    - Suggested limit: 5 SAR per day                                                                             │
│    - Justification: Lowering delivery spending to 5 SAR per day will help manage discretionary expenses more    │
│  effectively, as current spending is more than double this amount.                                              │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current spending: -90.50 SAR over 27 days                                                                  │
│    - Daily average: 90.50 SAR / 27 days ≈ 3.35 SAR                                                              │
│    - Suggested limit: 3 SAR per day                                                                             │
│    - Justification: Setting a limit of 3 SAR per day for transport will help maintain necessary travel while    │
│  keeping costs in check, as current spending is slightly above this limit.                                      │
│                                                                                                                 │
│  **2. Realistic Monthly Savings Target:**                                                                       │
│                                                                                                                 │
│  - Current total expenses: -3836.48 SAR                                                                         │
│  - Current remaining balance: 163.52 SAR                                                                        │
│  - Total income: 4000 SAR                              

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the verified spending data and shortfall projection below, build a                                       │
│  practical budget plan.                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  VERIFIED MONTHLY SPENDING SUMMARY (calculated directly from data):                                             │
│                                                                                                                 │
│  Income: 4000.00 SAR (received on day 1 of the month)                                                           │
│  Coffee: -435.50 SAR                                                                                            │
│  Delivery: -348.00 SAR                                                                                          │
│  Groceries: -973.50 SAR                                                                                         │
│  Bills & Rent: -1900.00 SAR                                                                                     │
│  Subscriptions: -88.98 SAR                                                                                      │
│  Transport: -90.50 SAR                                                                                          │
│                                                                                                                 │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Remaining balance: 163.52 SAR                                                                                  │
│                                                                                                                 │
│  Statement covers a 30-day month. Last recorded transaction was on day 27.                                      │
│  No transactions were recorded for days 28-30 (balance nearly depleted).                                        │
│                                                                                                                 │
│                                                                                                                 │
│  SHORTFALL PROJECTION (from the Predictor):                                                                     │
│  Step 1 - Daily spending rate:                                                                                  │
│  Total expenses: -3836.48 SAR                                                                                   │
│  Number of days covered: 27 days                                                                                │
│  Daily spending rate = Total expenses / Number of days covered                                                  │
│  Daily spending rate = -3836.48 SAR / 27 days = -142.06 SAR (approximately)                                     │
│                                                                                                                 │
│  Step 2 - Days the remaining balance can cover:                                                                 │
│  Remaining balance: 163.52 SAR                         

**Budget Plan**

**1. Suggested Daily Spending Limit for Discretionary Categories:**

- **Coffee:**
  - Current spending: -435.50 SAR over 27 days
  - Daily average: 435.50 SAR / 27 days ≈ 16.14 SAR
  - Suggested limit: 10 SAR per day
  - Justification: Reducing coffee spending to 10 SAR per day will help cut down on discretionary spending while still allowing for occasional coffee purchases, as current spending is significantly above this limit.

- **Delivery:**
  - Current spending: -348.00 SAR over 27 days
  - Daily average: 348.00 SAR / 27 days ≈ 12.89 SAR
  - Suggested limit: 5 SAR per day
  - Justification: Lowering delivery spending to 5 SAR per day will help manage discretionary expenses more effectively, as current spending is more than double this amount.

- **Transport:**
  - Current spending: -90.50 SAR over 27 days
  - Daily average: 90.50 SAR / 27 days ≈ 3.35 SAR
  - Suggested limit: 3 SAR per day
  - Justification: Setting a limit of 3 SAR per day for transport will help

In [35]:
# تحقق سريع من هدف الادخار (يُستخدم كمرجع نهائي بدل حساب الوكيل)
adjusted_expenses = 300 + 150 + 90 + 973.50 + 1900.00 + 88.98
verified_savings_target = 4000 - adjusted_expenses

print(f"Verified adjusted expenses: {adjusted_expenses:.2f} SAR")
print(f"Verified savings target: {verified_savings_target:.2f} SAR")

Verified adjusted expenses: 3502.48 SAR
Verified savings target: 497.52 SAR


In [40]:
# --- Task 6: Tone Rewrite Task ---
tone_task = Task(
    description=f'''
Below is a technical budget report. Do NOT translate or restructure it
section by section. Instead, completely rewrite it as a short, warm
message from Qirshak — as if a caring friend is texting the user, not
presenting a report.

{final_verified_report}

Write your rewrite as flowing conversational paragraphs (NOT bullet
points, NOT a numbered structure, NOT bold headers) that naturally cover:

- A warm, personal opening
- A specific, slightly humorous observation about coffee and delivery
  spending (use the real numbers, but phrase them like a friend would
  point them out, e.g. "قهوتك هالشهر وصلت X ريال — نحسبها استثمار ولا؟")
- A gentle, clear warning about the shortfall (which day, in plain words)
- The suggested daily limits, phrased as friendly suggestions, not a list
- An encouraging closing line mentioning the verified savings target

Use the VERIFIED figures section for the adjusted expenses and savings
target — do not use the original (possibly incorrect) numbers if they
differ.

Do not change any number, category name, or the shortfall day from the
source report — only change HOW it is said, not WHAT is said.

Write entirely in Arabic, in a warm Saudi conversational tone, like a
close friend texting — not a formal report. Maximum 150 words.
''',
    expected_output=(
        "A short, warm, conversational Arabic message (max 150 words, "
        "no bullet points or headers) in Qirshak's personal voice, "
        "covering the shortfall warning, spending highlights, daily "
        "limits, and verified savings target."
    ),
    agent=tone_agent,
    context=[budget_task]
)

In [41]:
tone_test_crew = Crew(
    agents=[tone_agent],
    tasks=[tone_task],
    process=Process.sequential,
    verbose=True
)

tone_result = await tone_test_crew.kickoff_async()
print(tone_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2d427c5d-9c2a-488a-8fd2-a1a6a96b453e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current spending: -435.50 SAR over 27 days                                                                 │
│    - Daily average: 435.50 SAR / 27 days ≈ 16.14 SAR                                                            │
│    - Suggested limit: 10 SAR per day                                                                            │
│    - Justification: Reducing coffee spending to 10 SAR per day will help cut down on discretionary spending     │
│  while still allowing for occasional coffee purchases, as current spending is significantly above this limit.   │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current spending: -348.00 SAR over 27 days                                                                 │
│    - Daily average: 348.00 SAR / 27 days ≈ 12.89 SAR                                                            │
│    - Suggested limit: 5 SAR per day                                                                             │
│    - Justification: Lowering delivery spending to 5 SAR per day will help manage discretionary expenses more    │
│  effectively, as current spending is more than double this amount.                                              │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current spending: -90.50 SAR over 27 days                                                                  │
│    - Daily average: 90.50 SAR / 27 days ≈ 3.35 SAR                                                              │
│    - Suggested limit: 3 SAR per day                                                                             │
│    - Justification: Setting a limit of 3 SAR per day for transport will help maintain necessary travel while    │
│  keeping costs in check, as current spending is slightly above this limit.                                      │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Qirshak Tone Rewriter                                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current spending: -435.50 SAR over 27 days                                                                 │
│    - Daily average: 435.50 SAR / 27 days ≈ 16.14 SAR                                                            │
│    - Suggested limit: 10 SAR per day                                                                            │
│    - Justification: Reducing coffee spending to 10 SAR per day will help cut down on discretionary spending     │
│  while still allowing for occasional coffee purchases, as current spending is significantly above this limit.   │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current spending: -348.00 SAR over 27 days                                                                 │
│    - Daily average: 348.00 SAR / 27 days ≈ 12.89 SAR                                                            │
│    - Suggested limit: 5 SAR per day                                                                             │
│    - Justification: Lowering delivery spending to 5 SAR per day will help manage discretionary expenses more    │
│  effectively, as current spending is more than double this amount.                                              │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current spending: -90.50 SAR over 27 days                                                                  │
│    - Daily average: 90.50 SAR / 27 days ≈ 3.35 SAR                                                              │
│    - Suggested limit: 3 SAR per day                                                                             │
│    - Justification: Setting a limit of 3 SAR per day for transport will help maintain necessary travel while    │
│  keeping costs in check, as current spending is slightl

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Qirshak Tone Rewriter                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على مدى 27 يوم،     │
│  يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة؟ لكن، لو قللناها لـ 10 ريال في اليوم، ممكن نخفف  │
│  من المصاريف الزائدة ونستمتع بالقهوة من وقت لآخر.                                                               │
│                                                                                                                 │
│  أما بالنسبة للتوصيل، فالوضع مو أفضل، 348 ريال في 27 يوم، يعني حوالي 12.89 ريال يومياً. لو حطينا حد 5 ريال في    │
│  اليوم، بنكون في السليم.                                                                                        │
│                                                                                                                 │
│  لا تنسى، باقي لك 163.52 ريال بس، وبتوصل لنهاية الشهر بعد 3 أيام!                                               │
│                                                                                                                 │
│  إذا مشينا على الحدود الجديدة، ممكن نوفر 497.52 ريال في الشهر. خلينا نبدأ مع بعض ونحقق هالهدف!                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Below is a technical budget report. Do NOT translate or restructure it                                         │
│  section by section. Instead, completely rewrite it as a short, warm                                            │
│  message from Qirshak — as if a caring friend is texting the user, not                                          │
│  presenting a report.                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  **Budget Plan**                                                                                                │
│                                                                                                                 │
│  **1. Suggested Daily Spending Limit for Discretionary Categories:**                                            │
│                                                                                                                 │
│  - **Coffee:**                                                                                                  │
│    - Current spending: -435.50 SAR over 27 days                                                                 │
│    - Daily average: 435.50 SAR / 27 days ≈ 16.14 SAR                                                            │
│    - Suggested limit: 10 SAR per day                                                                            │
│    - Justification: Reducing coffee spending to 10 SAR per day will help cut down on discretionary spending     │
│  while still allowing for occasional coffee purchases, as current spending is significantly above this limit.   │
│                                                                                                                 │
│  - **Delivery:**                                                                                                │
│    - Current spending: -348.00 SAR over 27 days                                                                 │
│    - Daily average: 348.00 SAR / 27 days ≈ 12.89 SAR                                                            │
│    - Suggested limit: 5 SAR per day                                                                             │
│    - Justification: Lowering delivery spending to 5 SAR per day will help manage discretionary expenses more    │
│  effectively, as current spending is more than double this amount.                                              │
│                                                                                                                 │
│  - **Transport:**                                                                                               │
│    - Current spending: -90.50 SAR over 27 days                                                                  │
│    - Daily average: 90.50 SAR / 27 days ≈ 3.35 SAR                                                              │
│    - Suggested limit: 3 SAR per day                                                                             │
│    - Justification: Setting a limit of 3 SAR per day for transport will help maintain necessary travel while    │
│  keeping costs in check, as current spending is slightly above this limit.                                      │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2d427c5d-9c2a-488a-8fd2-a1a6a96b453e                                                                       │
│  Final Output: يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على   │
│  مدى 27 يوم، يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة؟ لكن، لو قللناها لـ 10 ريال في       │
│  اليوم، ممكن نخفف من المصاريف الزائدة ونستمتع بالقهوة من وقت لآخر.                                              │
│                                                                                                                 │
│  أما بالنسبة للتوصيل، فالوضع مو أفضل، 348 ريال في 27 يوم، يعني حوالي 12.89 ريال يومياً. لو حطينا حد 5 ريال في    │
│  اليوم، بنكون في السليم.                                                                                        │
│                                                                                                                 │
│  لا تنسى، باقي لك 163.52 ريال بس، وبتوصل لنهاية الشهر بعد 3 أيام!                                               │
│                                                                                                                 │
│  إذا مشينا على الحدود الجديدة، ممكن نوفر 497.52 ريال في الشهر. خلينا نبدأ مع بعض ونحقق هالهدف!                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على مدى 27 يوم، يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة؟ لكن، لو قللناها لـ 10 ريال في اليوم، ممكن نخفف من المصاريف الزائدة ونستمتع بالقهوة من وقت لآخر. 

أما بالنسبة للتوصيل، فالوضع مو أفضل، 348 ريال في 27 يوم، يعني حوالي 12.89 ريال يومياً. لو حطينا حد 5 ريال في اليوم، بنكون في السليم. 

لا تنسى، باقي لك 163.52 ريال بس، وبتوصل لنهاية الشهر بعد 3 أيام! 

إذا مشينا على الحدود الجديدة، ممكن نوفر 497.52 ريال في الشهر. خلينا نبدأ مع بعض ونحقق هالهدف!




╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [42]:
# تصحيح لغوي بسيط لجملة العجز الملتبسة (بدون إعادة تشغيل الوكيل)
final_qirshak_message = tone_result.raw.replace(
    "وبتوصل لنهاية الشهر بعد 3 أيام",
    "وقدامك 3 أيام على نهاية الشهر — يعني ما راح يكفي"
)

print(final_qirshak_message)

يا صديقي، كيف حالك؟ حبيت أشارك معك شوية ملاحظات عن ميزانيتك. قهوتك هالشهر وصلت 435.50 ريال على مدى 27 يوم، يعني تقريباً 16.14 ريال في اليوم! هل نحسبها استثمار في السعادة؟ لكن، لو قللناها لـ 10 ريال في اليوم، ممكن نخفف من المصاريف الزائدة ونستمتع بالقهوة من وقت لآخر. 

أما بالنسبة للتوصيل، فالوضع مو أفضل، 348 ريال في 27 يوم، يعني حوالي 12.89 ريال يومياً. لو حطينا حد 5 ريال في اليوم، بنكون في السليم. 

لا تنسى، باقي لك 163.52 ريال بس، وقدامك 3 أيام على نهاية الشهر — يعني ما راح يكفي! 

إذا مشينا على الحدود الجديدة، ممكن نوفر 497.52 ريال في الشهر. خلينا نبدأ مع بعض ونحقق هالهدف!


In [ ]:
# --- Test Group 3 on its own ---
group3_crew = Crew(
    agents=[budget_advisor, tone_agent],
    tasks=[budget_task, tone_task],
    process=Process.sequential,
    verbose=True
)

group3_result = await group3_crew.kickoff_async()
print(group3_result.raw)

---
# Section 5: Assemble the Full Crew — Integration Owner Only

**Do not run these cells until every group's agents work successfully on their own.**

Merge steps:
1. Delete both placeholders (`placeholder_categorized_data`, `placeholder_full_analysis`)
2. Update `context=[...]` on each Task to link the real tasks together (not the placeholder text)
3. Order the six agents and tasks in the correct dependency order

In [ ]:
# TODO (Integration Owner):
# 1. Update alert_task and prediction_task to use:
#    context=[categorization_task, pattern_task]
#    instead of placeholder_categorized_data
#
# 2. Update budget_task to use:
#    context=[pattern_task, alert_task, prediction_task]
#    instead of placeholder_full_analysis

qirshak_crew = Crew(
    agents=[
        categorizer,
        pattern_detective,
        time_alert_agent,
        predictor,
        budget_advisor,
        tone_agent
    ],
    tasks=[
        categorization_task,
        pattern_task,
        alert_task,
        prediction_task,
        budget_task,
        tone_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Qirshak full crew (6 agents) assembled successfully.")

---
# Section 6: Final Run & Test

In [ ]:
final_result = await qirshak_crew.kickoff_async()
print(final_result.raw)

In [ ]:
# Display each agent's output separately (for review and the live demo)
labels = [
    "CATEGORIZER", "PATTERN DETECTIVE", "TIME-BASED ALERT",
    "PREDICTOR", "BUDGET & SAVINGS ADVISOR", "TONE AGENT (FINAL REPORT)"
]

for label, output in zip(labels, final_result.tasks_output):
    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    print(output.raw)